# Chapter 3 — Self-Attention Q&A Notes

Working notes from questions asked while going through `listing_3_1_class.py` / `listing_3_2_class.py` (simplified self-attention, Chapter 3 of *Build a Large Language Model From Scratch*).

Run this notebook from the `chapter_3/` folder so the local imports resolve.

In [ ]:
import torch
from listing_3_1_class import SelfAttention_v1
from listing_3_2_class import SelfAttention_v2

torch.manual_seed(123)

d_in = 3
d_out = 2

inputs = torch.tensor([[0.43, 0.15, 0.89],  # Your
                       [0.55, 0.87, 0.66],  # Journey
                       [0.57, 0.85, 0.64],  # starts
                       [0.22, 0.58, 0.33],  # with
                       [0.77, 0.25, 0.10],  # one
                       [0.05, 0.80, 0.55]]) # step

inputs.shape

## Q: What is the interpretation of the context vector?

`context_vec = attn_weights @ values` — each row is the context vector for one token.

It's the original token re-expressed as a **weighted blend of every token's `value` vector in the sequence** (including itself). The weights are the softmax attention weights: the more relevant another token is, the more its `value` contributes.

- **Input embedding** = the word's static, context-free meaning.
- **Context vector** = the word's meaning enriched by its surrounding sentence, weighted by attention.

This is exactly what lets the model disambiguate the same word differently depending on context.

In [ ]:
sa_v1 = SelfAttention_v1(d_in, d_out)
print("Context vectors:\n", sa_v1(inputs))

## Q: What is the dimension of the weight matrix?

`W_query`, `W_key`, `W_value` are each `(d_in, d_out)` = `(3, 2)` here.

For `x @ W` to work, where `x` is `(num_tokens, d_in)`, the matrix-multiply rule requires `W`'s first dimension to match `d_in`; the result's width becomes `W`'s last dimension (`d_out`):

```
(6, 3) @ (3, 2) = (6, 2)
```

Rows of `W` = `d_in` (input embedding size). Columns of `W` = `d_out` (projected size). The three matrices share this shape but are independently learned.

In [ ]:
print("W_query shape:", sa_v1.W_query.shape)
print(sa_v1.W_query)
print("\nW_key shape:", sa_v1.W_key.shape)
print(sa_v1.W_key)
print("\nW_value shape:", sa_v1.W_value.shape)
print(sa_v1.W_value)

## Q: What's the shape of a single token's query (e.g. `x_2` = "Journey")?

Depends entirely on *how you slice* the token — a 1D PyTorch tensor has **no row/column orientation** at all, unlike a classic linear-algebra column vector.

- `inputs[1]` → shape `(3,)`, a plain 1D tensor. `x_2 @ W_query` → shape `(2,)`, also 1D.
- `inputs[1:2]` → shape `(1, 3)`, explicitly 2D. `x_2 @ W_query` → shape `(1, 2)`.

Neither is "wrong"; `torch.matmul` has a special rule for 1D-by-2D multiplication that treats the 1D tensor as a row for the multiply, but returns a 1D result — not literally `2x1`.

In [ ]:
x_2 = inputs[1]
print("x_2 (1D):", x_2, x_2.shape)
q_2 = x_2 @ sa_v1.W_query
print("query_2 (1D):", q_2, q_2.shape)

x_2_row = inputs[1:2]
print("\nx_2_row (2D):", x_2_row, x_2_row.shape)
q_2_row = x_2_row @ sa_v1.W_query
print("query_2_row (2D):", q_2_row, q_2_row.shape)

## Q: How exactly is `query_2 = x_2 @ W_query` computed?

It's two dot products — one per column of `W_query`. Each output element sums 3 products (one per input dimension, `d_in=3`):

```
query_2[0] = x_2[0]*W_query[0,0] + x_2[1]*W_query[1,0] + x_2[2]*W_query[2,0]
query_2[1] = x_2[0]*W_query[0,1] + x_2[1]*W_query[1,1] + x_2[2]*W_query[2,1]
```

In [ ]:
manual_col0 = x_2[0]*sa_v1.W_query[0,0] + x_2[1]*sa_v1.W_query[1,0] + x_2[2]*sa_v1.W_query[2,0]
manual_col1 = x_2[0]*sa_v1.W_query[0,1] + x_2[1]*sa_v1.W_query[1,1] + x_2[2]*sa_v1.W_query[2,1]

print("manual query_2:", manual_col0.item(), manual_col1.item())
print("x_2 @ W_query: ", q_2)

## Q: During self-attention, do only the inputs change, while the weight matrices stay the same for the whole sentence?

Yes. Within one forward pass, `W_query`/`W_key`/`W_value` are **fixed and shared across every token**. Only `x_i` changes per token.

```
inputs (6, 3)  @  W_query (3, 2)  =  queries (6, 2)
  ↑ 6 different       ↑ same matrix      ↑ 6 different
    token vectors        for every row      query vectors
```

This weight sharing is what makes attention position-independent. The matrices only change across *training steps* (via backprop), not while processing a single sentence.

## Q: At the start, are all the weight matrices random?

Yes — `W_query`, `W_key`, `W_value` (and every other trainable matrix: embeddings, feed-forward, output layer) start as random numbers with zero learned information.

- Not "pure" naive random: PyTorch uses specific init schemes (`nn.Linear` defaults to Kaiming/uniform scaled by layer size; `torch.rand` used in `SelfAttention_v1` is plain uniform `[0,1)`).
- `torch.manual_seed(123)` doesn't remove randomness, it just makes it *reproducible* across runs.
- Meaningful structure (similar words → similar vectors, attention learning what to focus on) only emerges through training.

## Q: When calling `sa_v2.W_query(inputs)` (an `nn.Linear`), are we skipping the dot product? Is `nn.Linear` an improvement over it?

No — it's the *same* dot products, just packaged differently. `nn.Linear.__call__` internally computes `x @ weight.T (+ bias)`.

In [ ]:
import torch.nn as nn

torch.manual_seed(123)
lin = nn.Linear(3, 2, bias=False)
x = torch.tensor([0.55, 0.87, 0.66])

print("lin.weight shape (out_features, in_features):", lin.weight.shape)

out_via_call = lin(x)
out_via_matmul = x @ lin.weight.T

print("lin(x):       ", out_via_call)
print("x @ weight.T: ", out_via_matmul)
print("identical?", torch.allclose(out_via_call, out_via_matmul))

The real differences between `SelfAttention_v1` (`nn.Parameter` + manual `@`) and `SelfAttention_v2` (`nn.Linear`) are not mathematical:

1. **Weight storage convention** — `nn.Linear`'s weight is `(out_features, in_features)`, the transpose of v1's `(d_in, d_out)`. That's why `forward()` calls `self.W_key(x)` instead of `x @ self.W_key`.
2. **Better default initialization** — Kaiming-uniform (scaled by `1/√fan_in`) vs. plain `torch.rand(0,1)`.
3. **Optional bias term** — `qkv_bias=True` adds a learnable bias, which plain `nn.Parameter` in v1 doesn't have built in.
4. **Framework integration** — auto-registers as a submodule, shows up in `model.parameters()`, works with `.to(device)`, etc.

## Q: Without trainable matrices, I can see the interaction between words (dot product of embeddings). Where is that interaction with trainable matrices (`SelfAttention_v2`)?

Same mechanism, projected representations:

```
dot(x_i,  x_query)                        <- no weights: raw embedding vs raw embedding
dot(q_i,  k_j) = dot(x_i @ W_query, x_j @ W_key)   <- with weights: projected query vs projected key
```

The pairwise dot product (the actual "interaction") never goes away. What changes is that each token is first passed through a learned, role-specific lens before the comparison: `W_query` asks "what is this token looking for?", `W_key` asks "what does this token offer?".

A visible consequence: without weights, `dot(x_i, x_j)` is symmetric (same value either direction). With weights, `dot(q_i, k_j) != dot(q_j, k_i)` in general, because `W_query != W_key`.

In [ ]:
from listing_3_2_class import SelfAttention_v2

print('=== WITHOUT trainable weights ===')
query_raw = inputs[1]
scores_raw = torch.tensor([torch.dot(x_i, query_raw) for x_i in inputs])
print('scores_raw:', scores_raw)
print('symmetric? dot(x_1,x_4) vs dot(x_4,x_1):',
      torch.dot(inputs[1], inputs[4]).item(), 'vs', torch.dot(inputs[4], inputs[1]).item())

print()
print('=== WITH trainable weights ===')
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in=3, d_out=2)
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)

print('asymmetric? dot(q_1,k_4) vs dot(q_4,k_1):',
      torch.dot(queries[1], keys[4]).item(), 'vs', torch.dot(queries[4], keys[1]).item())

## Q: Full worked example for a single token — `x_2` ("Journey"). What is `c_2`?

Notation: `x_j` = every token in the sentence (`j=1..6`). Every `x_j` becomes a `key_j` and a `value_j`. Only `x_2` becomes the `query` we use here, since we're building the context vector for token 2.

| j | token `x_j` | `key_j` | `value_j` | `score_2j = query_2 · key_j` | `weight_2j = softmax(score_2j)` |
|---|---|---|---|---|---|
| 1 | Your | [0.3147, -0.4016] | [-0.0872, 0.0286] | 0.4656 | **0.2041** |
| 2 | Journey | [-0.0298, -0.4459] | [-0.1137, 0.0766] | 0.1723 | 0.1659 |
| 3 | starts | [-0.0170, -0.4262] | [-0.1018, 0.0927] | 0.1751 | 0.1662 |
| 4 | with | [-0.1054, -0.2724] | [-0.0912, -0.0026] | 0.0259 | 0.1496 |
| 5 | one | [0.2185, 0.0482] | [0.1395, 0.3580] | 0.1771 | 0.1665 |
| 6 | step | [-0.2258, -0.4782] | [-0.2085, -0.1546] | 0.0085 | 0.1477 |

(`query_2 = [0.9091, -0.4471]`, fixed for the whole row — it's "Journey"'s own query, compared against every `key_j`.)

Final step — `c_2` is the weighted sum of all 6 `value_j`, using the weights above:

```
c_2 = 0.2041*value_1 + 0.1659*value_2 + 0.1662*value_3 + 0.1496*value_4 + 0.1665*value_5 + 0.1477*value_6
    = [-0.0748, 0.0703]
```

In [ ]:
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)
query_2 = sa_v2.W_query(inputs[1])

scores_2 = query_2 @ keys.T
weights_2 = torch.softmax(scores_2 / keys.shape[-1]**0.5, dim=-1)
c_2 = weights_2 @ values

print('query_2:', query_2)
print('scores_2:', scores_2)
print('weights_2:', weights_2, ' sum:', weights_2.sum())
print('c_2:', c_2)

## Q: Are attention *scores* and attention *weights* symmetric, with or without trainable matrices?

Important correction/clarification: **only the scores can be symmetric, never the weights** — even in the no-weights case.

- **Scores** `dot(x_i, x_j)`: symmetric *only* when there are no trainable matrices (same two raw vectors, same dot product either direction). With `W_query != W_key`, scores become asymmetric too (`dot(q_i,k_j) != dot(q_j,k_i)`), as shown above.
- **Weights** (after softmax): **never symmetric**, regardless of trainable matrices or not. Each row of the attention matrix is normalized independently (`softmax(..., dim=-1)`), and each row sums over a *different* set of scores — so row `i`'s normalizing denominator differs from row `j`'s, even when the raw scores matrix itself is perfectly symmetric.

In [ ]:
# No trainable weights, full 6x6 matrix
scores_full = inputs @ inputs.T
weights_full = torch.softmax(scores_full, dim=-1)

print('scores symmetric?', torch.allclose(scores_full, scores_full.T))
print('weights symmetric?', torch.allclose(weights_full, weights_full.T))
print('weights[1,4]:', weights_full[1,4].item(), ' weights[4,1]:', weights_full[4,1].item())

## Q: Where does the concatenation in `MultiHeadAttentionWrapper` come from, and why?

`listing_3_4.py`: `torch.cat([head(x) for head in self.heads], dim=-1)`. With `num_heads=2` and `d_out=2`, each head is a fully separate `CausalAttention` instance with its own independently-initialized `W_query`/`W_key`/`W_value`. Both heads process the *same* input independently, producing their own `(batch, num_tokens, d_out)` context vectors. `torch.cat(..., dim=-1)` glues each head's output onto the end of the previous one.

For token 0 ("Your"): `head 1 -> [-0.4519, 0.2216]`, `head 2 -> [0.4772, 0.1063]`, concatenated -> `[-0.4519, 0.2216, 0.4772, 0.1063]` — exactly the 4-number row you see.

**Why bother?** Each head can, after training, end up focusing on different relationships between tokens, so concatenating lets the model carry multiple "views" of the sentence forward together instead of being limited to one.

**Important correction:** we can't predict in advance *which* head will specialize in what (e.g. "head 1 = short-range, head 2 = long-range" is not something the architecture guarantees). At random initialization, neither head has any structure yet — specialization, if it happens at all, is an emergent, unplanned side effect of training, only observable afterward by inspecting real attention-weight patterns on real sentences.

In [ ]:
from listing_3_4 import MultiHeadAttentionWrapper

torch.manual_seed(123)
batch2 = torch.stack((inputs, inputs), dim=0)
context_length2 = batch2.shape[1]

mha = MultiHeadAttentionWrapper(d_in=3, d_out=2, context_length=context_length2, dropout=0.0, num_heads=2)

head1_out = mha.heads[0](batch2)
head2_out = mha.heads[1](batch2)
print('head 1 output for token 0:', head1_out[0, 0])
print('head 2 output for token 0:', head2_out[0, 0])
print('concatenated:', torch.cat([head1_out, head2_out], dim=-1)[0, 0])

print()
print('--- attn_weights per head, sample 0 (shows NO specialization yet, random init) ---')
for i, head in enumerate(mha.heads):
    keys = head.W_key(batch2)
    queries = head.W_query(batch2)
    scores = queries @ keys.transpose(1, 2)
    scores = scores.masked_fill(head.mask.bool()[:context_length2, :context_length2], -torch.inf)
    weights = torch.softmax(scores / keys.shape[-1]**0.5, dim=-1)
    print(f'head {i}:')
    print(weights[0].detach().round(decimals=2))

## Q: Is the context vector important, or should I focus on the attention weights?

- **Attention weights** — a probability distribution (0 to 1, each row summing to 1) saying "how much should this token borrow from each other token." The *recipe*. Useful for interpretability (visualizing what the model attends to), but not consumed directly by later layers.
- **Context vector** — the actual result: each token's new representation after blending in the other tokens' `value` vectors using those weights. The *dish* — this is what flows forward into the rest of the network (next transformer block, feed-forward layer, output layer, etc.).

**So: the context vector is the important variable computationally.** Attention weights are the mechanism that produces it, and mainly matter for *your own* understanding/debugging of the model.

## Q: What does a context vector represent? Take `z(0)` as an example.

Using the `MultiHeadAttentionWrapper` example: `z(0) = [-0.4519, 0.2216, 0.4772, 0.1063]`.

In general, `z(i)` = token `i`'s meaning, rebuilt as a weighted blend of the `value` vectors of every token it's allowed to attend to (itself + everything before it, due to the causal mask).

`z(0)` is a special, degenerate case: token 0 ("Your") is first, so there's nothing before it — its attention row is `[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]` (100% on itself). That means:

```
z(0) = 1.0 * value(0) + 0.0 * value(1) + ... = value(0)
```

`z(0)` is literally just `value(0)` — no real context-mixing happened, because none was available yet. Compare with `z(5)` ("step", the last token), whose attention row spreads roughly evenly across all 6 tokens (`~0.17` each) — a genuine blend of the whole sentence so far.

**Dimension of `z(0)`:** confirmed `4` = `num_heads (2) * d_out (2)`, via `Context Vectors shape: torch.Size([2, 6, 4])`. Every token's context vector has this same width, regardless of position.

## Q: Isn't computing the context vector just a scalar (dot) product? Why do we get a vector back?

It *is* built from scalar dot products — just one dot product per output dimension, not a single overall one.

`c_i = attn_weights[i] @ values`, where `attn_weights[i]` has shape `(num_tokens,)` and `values` has shape `(num_tokens, d_out)`:

```
c_i[0] = attn_weights[i,0]*values[0,0] + attn_weights[i,1]*values[1,0] + ...   <- one scalar dot product (column 0)
c_i[1] = attn_weights[i,0]*values[0,1] + attn_weights[i,1]*values[1,1] + ...   <- a SECOND scalar dot product (column 1)
```

Each individual dot product does collapse to a scalar — but `values` has `d_out` columns, so the dot product repeats once per column, and the results stack back into a `d_out`-length vector. Same pattern as `query_2 = x_2 @ W_query` earlier in this notebook: one scalar per column of the weight matrix, stacked into a vector.